# Notebook 02 · Optimización e hiperparámetros (SGD, RMSprop, Adam y learning rate)

| Redes Neuronales III | 604013/DEEP LEARNING -CONCEPTOS/FU/CAD2202023205/EIAIIPA2026 |
|---|---|
| Universidad de Cundinamarca · Especialización en Inteligencia Artificial | Docente: **Nidia Stella Garcia Roa** |
| Estudiante: **Yohann Javier Nikolay Salazar Cardenas** | Repositorio: `https://github.com/Yohann0011/Redes-Neuronales-III` |


**Problema:** el mismo del Notebook 01 (MNIST, 10 clases, red 784 → 128 → 10).

> **Alcance:** comparar tres optimizadores —**SGD con momentum**, **RMSprop** y **Adam**— manteniendo
> fijos dataset, arquitectura, épocas y batch size, y explorar el efecto del **learning rate** de Adam.
> La comparación se decide sobre **Dev**, nunca sobre Test.

---

## 1. Qué hace un optimizador

El descenso de gradiente indica la dirección; el optimizador decide el paso. Las tres familias usadas:

| Optimizador | Idea | Qué vigilar |
|---|---|---|
| **SGD + momentum** | Acumula una fracción de los pasos anteriores para conservar dirección | Suele exigir más ajuste del learning rate |
| **RMSprop** | Divide el paso por la raíz del promedio reciente de gradientes al cuadrado | Se adapta a la escala de cada parámetro |
| **Adam** | Combina momentum y adaptación por segundo momento | Buen punto de partida, no ganador universal |

**Regla metodológica:** para que la comparación atribuya las diferencias al optimizador, todo lo demás
debe permanecer idéntico: mismos datos, misma arquitectura, mismas épocas, mismo batch y misma semilla.

## 2. Entorno y datos (idénticos al Notebook 01)

In [1]:
import os
# Ambiente silencioso y determinista para TensorFlow (no altera los resultados)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["PYTHONHASHSEED"] = "0"

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.dpi": 80,
    "savefig.dpi": 80,
    "savefig.bbox": None,
    "figure.autolayout": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 9,
})
plt.rcParams["font.family"] = "DejaVu Sans"

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
import random, gc

random.seed(42)
tf.random.set_seed(42)
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)

I0000 00:00:1791728293.984632     480 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1791728296.093301     480 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow: 2.21.0 | Keras: 3.15.1


In [2]:
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

(x_train_full, y_train_full), (x_test_raw, y_test) = keras.datasets.mnist.load_data()

# División estratificada Train / Dev (80 / 20). Test queda intocable hasta el final.
x_train, x_dev, y_train, y_dev = train_test_split(
    x_train_full, y_train_full,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train_full,
)

# Test se normaliza completo (31 MB); Train y Dev se normalizan por lotes para
# mantener el consumo de memoria del kernel bajo control.
x_test = x_test_raw.astype("float32") / 255.0


def normalizar_lote(imagenes):
    """Convierte un lote uint8 (n, 28, 28) en float32 (n, 28, 28) dentro de [0, 1].

    El aplanado a 784 características lo realiza la capa Flatten del modelo.
    """
    return imagenes.astype("float32") / 255.0


print("Train:", x_train.shape, "| Dev:", x_dev.shape, "| Test:", x_test.shape)


Train: (48000, 28, 28) | Dev: (12000, 28, 28) | Test: (10000, 28, 28)


## 3. Función de modelo y de entrenamiento reutilizables

`crear_modelo_experimento` construye siempre la misma arquitectura; solo cambia el optimizador que
recibe. El entrenamiento usa el ciclo explícito por lotes del Notebook 01, encapsulado para poder
repetirlo en cada configuración.

In [3]:
def crear_modelo_experimento(optimizador, units=128):
    """Arquitectura fija 784 -> 128 -> 10 para comparar en igualdad de condiciones."""
    modelo = keras.Sequential([
        layers.Input(shape=(28, 28)),
        layers.Flatten(),
        layers.Dense(units, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])
    modelo.compile(
        optimizer=optimizador,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return modelo


def evaluar_modelo(modelo, x_datos, y_datos, tam_lote=1000):
    perdida_fn = keras.losses.SparseCategoricalCrossentropy()
    perdida_total, aciertos, total = 0.0, 0, 0
    for inicio in range(0, len(x_datos), tam_lote):
        xb = x_datos[inicio:inicio + tam_lote]
        yb = y_datos[inicio:inicio + tam_lote]
        xb = normalizar_lote(xb) if xb.dtype == np.uint8 else xb
        probs = modelo(xb, training=False).numpy()
        perdida_total += float(perdida_fn(yb, probs)) * len(yb)
        aciertos += int((probs.argmax(axis=1) == yb).sum())
        total += len(yb)
    return perdida_total / total, aciertos / total


def entrenar_modelo(modelo, epochs, batch_size=64, semilla=42):
    """Ciclo explícito por lotes; devuelve el historial de loss/accuracy en Train y Dev."""
    tf.random.set_seed(semilla)
    perdida_fn = keras.losses.SparseCategoricalCrossentropy()
    optimizador = modelo.optimizer

    @tf.function
    def paso(xb, yb):
        with tf.GradientTape() as cinta:
            logits = modelo(xb, training=True)
            perdida = perdida_fn(yb, logits)
        g = cinta.gradient(perdida, modelo.trainable_variables)
        optimizador.apply_gradients(zip(g, modelo.trainable_variables))
        return perdida, tf.argmax(logits, axis=1)

    indices = np.arange(len(x_train))
    hist = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
    for _ in range(epochs):
        np.random.shuffle(indices)
        suma, aciertos, vistos = 0.0, 0, 0
        for inicio in range(0, len(indices), batch_size):
            idx = indices[inicio:inicio + batch_size]
            xb = normalizar_lote(x_train[idx])
            yb = y_train[idx]
            perdida, pred = paso(xb, yb)
            suma += float(perdida) * len(idx)
            aciertos += int((pred.numpy() == yb).sum())
            vistos += len(idx)
        val_loss, val_acc = evaluar_modelo(modelo, x_dev, y_dev)
        hist["loss"].append(suma / vistos)
        hist["accuracy"].append(aciertos / vistos)
        hist["val_loss"].append(val_loss)
        hist["val_accuracy"].append(val_acc)
        gc.collect()
    return hist


print("Funciones de experimentación definidas correctamente")

Funciones de experimentación definidas correctamente


## 4. Comparación de optimizadores (5 épocas)

Cada configuración se entrena desde cero (`clear_session` + semilla) con el mismo ciclo. La tabla
resultante reporta la métrica de la **última época** en Train y Dev.

In [4]:
fabricas_optimizadores = {
    "SGD_momentum": lambda: keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    "RMSprop": lambda: keras.optimizers.RMSprop(learning_rate=0.001),
    "Adam": lambda: keras.optimizers.Adam(learning_rate=0.001),
}

EPOCHS_OPT = 5
resultados_opt = []
historias_opt = {}

for nombre, fabrica in fabricas_optimizadores.items():
    keras.backend.clear_session()
    gc.collect()
    modelo = crear_modelo_experimento(fabrica())
    hist = entrenar_modelo(modelo, epochs=EPOCHS_OPT)
    historias_opt[nombre] = hist
    resultados_opt.append({
        "optimizador": nombre,
        "train_acc": round(hist["accuracy"][-1], 4),
        "dev_acc": round(hist["val_accuracy"][-1], 4),
        "dev_loss": round(hist["val_loss"][-1], 4),
    })
    print(f"{nombre:<14} dev_loss {hist['val_loss'][-1]:.4f} | dev_acc {hist['val_accuracy'][-1]:.4f}")

tabla_opt = pd.DataFrame(resultados_opt)
print()
print("=== TABLA COMPARATIVA DE OPTIMIZADORES (época 5) ===")
display(tabla_opt)
print(tabla_opt.to_string(index=False))

SGD_momentum   dev_loss 0.1293 | dev_acc 0.9627


RMSprop        dev_loss 0.0964 | dev_acc 0.9723


Adam           dev_loss 0.0966 | dev_acc 0.9698

=== TABLA COMPARATIVA DE OPTIMIZADORES (época 5) ===


,optimizador,train_acc,dev_acc,dev_loss
0,SGD_momentum,0.9672,0.9627,0.1293
1,RMSprop,0.9800,0.9723,0.0964
2,Adam,0.9811,0.9698,0.0966


 optimizador  train_acc  dev_acc  dev_loss
SGD_momentum     0.9672   0.9627    0.1293
     RMSprop     0.9800   0.9723    0.0964
        Adam     0.9811   0.9698    0.0966


In [5]:
fig, ax = plt.subplots(figsize=(7.5, 4))
colores = {"SGD_momentum": "#c0392b", "RMSprop": "#2980b9", "Adam": "#27ae60"}
for nombre, hist in historias_opt.items():
    ax.plot(range(1, EPOCHS_OPT + 1), hist["val_loss"], marker="o",
            label=nombre, color=colores[nombre], lw=2)
ax.set_title("Figura 3. Validation loss por optimizador (misma arquitectura y semilla)")
ax.set_xlabel("Época"); ax.set_ylabel("Validation loss"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("figIII_02_optim_valloss.png"); plt.show()
plt.close(fig); _ = gc.collect()

## 5. Exploración del learning rate de Adam

El learning rate es el hiperparámetro más sensible: controla el tamaño del paso. Se prueban tres
órdenes de magnitud (1e-4, 1e-3, 1e-2) con 3 épocas cada uno. La lectura esperada: un paso demasiado
pequeño aprende lento; uno demasiado grande oscila o diverge.

In [6]:
learning_rates = [0.0001, 0.001, 0.01]
resultados_lr = []

for lr in learning_rates:
    keras.backend.clear_session()
    gc.collect()
    modelo = crear_modelo_experimento(keras.optimizers.Adam(learning_rate=lr))
    hist = entrenar_modelo(modelo, epochs=3)
    resultados_lr.append({
        "learning_rate": lr,
        "train_acc": round(hist["accuracy"][-1], 4),
        "dev_acc": round(hist["val_accuracy"][-1], 4),
        "dev_loss": round(hist["val_loss"][-1], 4),
    })

tabla_lr = pd.DataFrame(resultados_lr)
print("=== EFECTO DEL LEARNING RATE (Adam, 3 épocas) ===")
display(tabla_lr)
print(tabla_lr.to_string(index=False))

=== EFECTO DEL LEARNING RATE (Adam, 3 épocas) ===


,learning_rate,train_acc,dev_acc,dev_loss
0,0.0001,0.9170,0.9213,0.2851
1,0.0010,0.9686,0.9675,0.1099
2,0.0100,0.9673,0.9595,0.1583


 learning_rate  train_acc  dev_acc  dev_loss
        0.0001     0.9170   0.9213    0.2851
        0.0010     0.9686   0.9675    0.1099
        0.0100     0.9673   0.9595    0.1583


In [7]:
fig, ax = plt.subplots(figsize=(7.0, 3.8))
ax.plot([r["learning_rate"] for r in resultados_lr],
        [r["dev_acc"] for r in resultados_lr], marker="s", lw=2, color="#8e44ad")
for r in resultados_lr:
    ax.annotate(f'{r["dev_acc"]:.4f}', (r["learning_rate"], r["dev_acc"]),
                textcoords="offset points", xytext=(0, 9), ha="center", fontsize=9)
ax.set_xscale("log")
ax.set_xticks(learning_rates)
ax.set_xticklabels([str(lr) for lr in learning_rates])
ax.set_xlabel("Learning rate (Adam)")
ax.set_ylabel("Dev accuracy")
ax.set_title("Figura 4. Dev accuracy frente al learning rate")
plt.tight_layout(); plt.savefig("figIII_03_lr_dev.png"); plt.show()
plt.close(fig); _ = gc.collect()

## 6. Hiperparámetros: qué controla cada uno

| Hiperparámetro | Qué controla | Riesgo si es extremo |
|---|---|---|
| Learning rate | Tamaño del paso de actualización | Alto: inestabilidad y divergencia. Bajo: aprendizaje lento |
| Batch size | Ejemplos procesados antes de actualizar | Cambia el ruido del gradiente, la memoria y la velocidad |
| Neuronas por capa | Capacidad de representación | Pocas: underfitting. Muchas: costo y riesgo de overfitting |
| Épocas | Pasadas completas por Train | Pocas: aprendizaje insuficiente. Muchas: sobreajuste |

## 7. Análisis y conclusiones

- La comparación de optimizadores **no admite conclusiones universales**: la afirmación correcta es
  «bajo las condiciones de este experimento (MNIST, red densa, 5 épocas, semilla 42), tal optimizador
  mostró mejor o peor comportamiento en Dev».
- Adam con lr = 0.001 suele converger rápido en pocas épocas; SGD con momentum y lr = 0.01 puede
  alcanzar resultados similares con más épocas; RMSprop se ubica habitualmente entre ambos.
- La curva de validation loss es el criterio de comparación más informativo: muestra no solo el punto
  final sino la **estabilidad** del descenso.
- El learning rate se explora en escala logarítmica porque su efecto es multiplicativo: pasar de
  1e-4 a 1e-3 cambia el paso diez veces, no una unidad.

## 8. Referencias

- Gulli, A., Kapoor, A., Pal, S., & Ciraci, S. (2023). *Deep Learning with TensorFlow and Keras* (3rd ed.). Packt Publishing.
- Trask, A. (2019). *Grokking Deep Learning*. Manning Publications.
- Google (2026). *Google Colaboratory*. https://colab.research.google.com/